In [1]:
import os, torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

HF_TOKEN = "Your Key Here" 
BASE_MODEL   = "meta-llama/Llama-3.2-3B-Instruct"
DATASET_NAME = "text2sqldata"
HUB_OUT      = "text2sql-llama3-qlora"
RUN_NAME     = "text2sql-run-1"
EPOCHS       = 3
LR           = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
MAX_SEQ_LEN  = 512
SAVE_STEPS   = 100
LOG_STEPS    = 10
LORA_R       = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.05

login(token=HF_TOKEN)
print(" Ready!")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

 Ready!
GPU  : NVIDIA L40
VRAM : 47.7 GB


In [2]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    trust_remote_code=True
)

# Use token ID 1 as pad — safe within all vocab boundaries
tokenizer.pad_token_id = 1
tokenizer.pad_token    = tokenizer.convert_ids_to_tokens(1)
tokenizer.padding_side = "right"


print(f"\nVOCABULARY INFO:")
print(f"   Vocab size      : {tokenizer.vocab_size:,} tokens")
print(f"   Special tokens  : {tokenizer.all_special_tokens}")

print(f"\nTOKEN IDs:")
print(f"   BOS (start)     : {tokenizer.bos_token} = ID {tokenizer.bos_token_id}")
print(f"   EOS (end)       : {tokenizer.eos_token} = ID {tokenizer.eos_token_id}")
print(f"   PAD (padding)   : {tokenizer.pad_token} = ID {tokenizer.pad_token_id}")

print(f"\nTOKENIZATION EXAMPLE:")
test_text = "How many electric bike trips in July?"
tokens = tokenizer(test_text)
print(f"   Input text      : '{test_text}'")
print(f"   Token IDs       : {tokens['input_ids']}")
print(f"   Decoded back    : '{tokenizer.decode(tokens['input_ids'])}'")
print(f"   Num tokens      : {len(tokens['input_ids'])}")

print(f"\nEACH TOKEN DECODED:")
for tid in tokens['input_ids']:
    print(f"   {tid:6d} → '{tokenizer.decode([tid])}'")


VOCABULARY INFO:
   Vocab size      : 128,000 tokens
   Special tokens  : ['<|begin_of_text|>', '<|eot_id|>', '"']

TOKEN IDs:
   BOS (start)     : <|begin_of_text|> = ID 128000
   EOS (end)       : <|eot_id|> = ID 128009
   PAD (padding)   : " = ID 1

TOKENIZATION EXAMPLE:
   Input text      : 'How many electric bike trips in July?'
   Token IDs       : [128000, 4438, 1690, 9249, 13260, 23277, 304, 5887, 30]
   Decoded back    : '<|begin_of_text|>How many electric bike trips in July?'
   Num tokens      : 9

EACH TOKEN DECODED:
   128000 → '<|begin_of_text|>'
     4438 → 'How'
     1690 → ' many'
     9249 → ' electric'
    13260 → ' bike'
    23277 → ' trips'
      304 → ' in'
     5887 → ' July'
       30 → '?'


In [4]:
# Loading Model with Complete Summary
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch


# Configure 4-bit quantization
print("\nConfiguring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
print("   load_in_4bit=True, compress weights to 4-bit")
print("   bnb_4bit_quant_type=nf4, NormalFloat4 quantization")
print("   use_double_quant=True, extra compression layer")
print("   compute_dtype=float16, use float16 for calculations")

# Load model
print(f"\n Downloading {BASE_MODEL}...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
    attn_implementation="eager"
)

# pad token
model.config.pad_token_id            = 1
model.generation_config.pad_token_id = 1
model.config.use_cache               = False

print("\n Model loaded!")

# Complete model summary
print("MODEL SUMMARY")


print(f"\nBASIC INFO:")
print(f"   Model name     : {model.config._name_or_path}")
print(f"   Model type     : {model.config.model_type}")
print(f"   Architecture   : {model.__class__.__name__}")

print(f"\nARCHITECTURE DIMENSIONS:")
print(f"   Vocab size     : {model.config.vocab_size:,} tokens")
print(f"   Hidden size    : {model.config.hidden_size:,}")
print(f"   (embedding vector size per token)")
print(f"   Num layers     : {model.config.num_hidden_layers}")
print(f"   (transformer blocks)")
print(f"   Num attn heads : {model.config.num_attention_heads}")
print(f"   (parallel attention heads)")
print(f"   KV heads       : {model.config.num_key_value_heads}")
print(f"   FFN hidden size: {model.config.intermediate_size:,}")
print(f"   Max position   : {model.config.max_position_embeddings:,} tokens")

print(f"\nMEMORY:")
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total params   : {total_params/1e9:.2f}B ({total_params:,})")
print(f"   Trainable      : {trainable:,}")
print(f"   Frozen         : {total_params - trainable:,}")
print(f"   Memory (4-bit) : {model.get_memory_footprint()/1e9:.2f} GB")
print(f"   Memory (full)  : ~{total_params * 4 / 1e9:.1f} GB (if float32)")
print(f"   Memory saved   : {(1 - model.get_memory_footprint()/(total_params*4))*100:.0f}% saved by 4-bit!")

print(f"\nMODEL LAYERS:")
print(f"   embed_tokens   : {model.config.vocab_size:,} × {model.config.hidden_size:,}")
print(f"   (input embedding table)")
for i in range(min(3, model.config.num_hidden_layers)):
    print(f"   Layer {i}        : attention + FFN + norms")
print(f"   ... {model.config.num_hidden_layers - 3} more layers ...")
print(f"   lm_head        : {model.config.hidden_size:,} × {model.config.vocab_size:,}")
print(f"   (output prediction layer)")

print(f"\nTRAINING CONFIG:")
print(f"   Pad token id   : {model.config.pad_token_id} ← safe! (< 128,000)")
print(f"   use_cache      : {model.config.use_cache} ← disabled for training")
print(f"   Attention impl : eager ← no SDPA bugs")
print(f"   Compute dtype  : float16")

print(f"\nGPU STATUS:")
print(f"   GPU            : {torch.cuda.get_device_name(0)}")
print(f"   Total VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"   Used VRAM      : {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"   Free VRAM      : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.1f} GB")
print(f"   Model on device: {next(model.parameters()).device}")

print(f"   Model ready for LoRA fine-tuning!")


Configuring 4-bit quantization...
   load_in_4bit=True, compress weights to 4-bit
   bnb_4bit_quant_type=nf4, NormalFloat4 quantization
   use_double_quant=True, extra compression layer
   compute_dtype=float16, use float16 for calculations



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


 Model loaded!
MODEL SUMMARY

BASIC INFO:
   Model name     : meta-llama/Llama-3.2-3B-Instruct
   Model type     : llama
   Architecture   : LlamaForCausalLM

ARCHITECTURE DIMENSIONS:
   Vocab size     : 128,256 tokens
   Hidden size    : 3,072
   (embedding vector size per token)
   Num layers     : 28
   (transformer blocks)
   Num attn heads : 24
   (parallel attention heads)
   KV heads       : 8
   FFN hidden size: 8,192
   Max position   : 131,072 tokens

MEMORY:
   Total params   : 1.80B (1,803,463,680)
   Trainable      : 394,177,536
   Frozen         : 1,409,286,144
   Memory (4-bit) : 2.20 GB
   Memory (full)  : ~7.2 GB (if float32)
   Memory saved   : 70% saved by 4-bit!

MODEL LAYERS:
   embed_tokens   : 128,256 × 3,072
   (input embedding table)
   Layer 0        : attention + FFN + norms
   Layer 1        : attention + FFN + norms
   Layer 2        : attention + FFN + norms
   ... 25 more layers ...
   lm_head        : 3,072 × 128,256
   (output prediction layer)

TRAINI

In [6]:
# Load Dataset and Understanding
from datasets import load_dataset

# Load from HuggingFace
print("\n Loading dataset from HuggingFace...")
ds = load_dataset(DATASET_NAME)

print(f"\n Dataset loaded!")
print("DATASET OVERVIEW")
print(f"\n SIZE:")
print(f"   Train examples : {len(ds['train']):,}")
print(f"   Val examples   : {len(ds['validation']):,}")
print(f"   Total          : {len(ds['train']) + len(ds['validation']):,}")

print(f"\n COLUMNS:")
for col in ds['train'].column_names:
    print(f"   → {col}")


print("ONE COMPLETE TRAINING EXAMPLE")
example = ds['train'][0]

print(f"\n SOURCE: {example['source']}")
print(f"   (spider = general SQL | bikeshare = my domain)")

print(f"\n DB_ID: {example['db_id']}")
print(f"   (which database this example belongs to)")

print(f"\n QUESTION:")
print(f"   {example['question']}")

print(f"\n SQL (correct answer):")
print(f"   {example['sql']}")

print(f"\n FULL TEXT (what model sees):")
print(example['text'])

print("SOURCE DISTRIBUTION")


train_sources = ds['train']['source']
val_sources   = ds['validation']['source']

spider_train = train_sources.count('spider')
bike_train   = train_sources.count('bikeshare')
spider_val   = val_sources.count('spider')
bike_val     = val_sources.count('bikeshare')

print(f"\n   TRAIN SET ({len(ds['train']):,} examples):")
print(f"   Spider    : {spider_train:,} ({spider_train/len(ds['train'])*100:.1f}%)")
print(f"             , teaches general SQL across 200 databases")
print(f"   Bikeshare : {bike_train:,} ({bike_train/len(ds['train'])*100:.1f}%)")
print(f"             , teaches our specific schema")

print(f"\n   VAL SET ({len(ds['validation']):,} examples):")
print(f"   Spider    : {spider_val:,} ({spider_val/len(ds['validation'])*100:.1f}%)")
print(f"   Bikeshare : {bike_val:,} ({bike_val/len(ds['validation'])*100:.1f}%)")

print("DIFFICULTY DISTRIBUTION (bikeshare examples)")

bike_examples = [ex for ex in ds['train'] if ex['source'] == 'bikeshare']

easy   = [ex for ex in bike_examples if 'JOIN' not in ex['sql'].upper() 
          and 'GROUP BY' not in ex['sql'].upper()
          and 'SELECT' in ex['sql'].upper()]
medium = [ex for ex in bike_examples if 'JOIN' in ex['sql'].upper() 
          or 'GROUP BY' in ex['sql'].upper()]
hard   = [ex for ex in bike_examples if 'OVER' in ex['sql'].upper() 
          or 'PARTITION' in ex['sql'].upper()
          or ex['sql'].upper().count('SELECT') > 1]

print(f"\n   Easy   (simple SELECT/WHERE) : ~{len(easy):,}")
print(f"   Medium (JOINs, GROUP BY)     : ~{len(medium):,}")
print(f"   Hard   (subqueries, window)  : ~{len(hard):,}")


print("TOKEN LENGTH ANALYSIS")


print("\n   Tokenizing 100 samples to check lengths...")
lengths = []
for i in range(min(100, len(ds['train']))):
    tokens = tokenizer(
        ds['train'][i]['text'],
        add_special_tokens=False
    )
    lengths.append(len(tokens['input_ids']))

import numpy as np
print(f"\n   Min tokens  : {min(lengths)}")
print(f"   Max tokens  : {max(lengths)}")
print(f"   Avg tokens  : {sum(lengths)//len(lengths)}")
print(f"   Our MAX_SEQ : {MAX_SEQ_LEN}")
print(f"   Fits in 512 : {sum(1 for l in lengths if l <= MAX_SEQ_LEN)}/100 examples")


print("BIKESHARE DOMAIN EXAMPLE")

bike_ex = next(ex for ex in ds['train'] if ex['source'] == 'bikeshare')
print(f"\n QUESTION : {bike_ex['question']}")
print(f" SQL      : {bike_ex['sql']}")
print(f"\n FULL PROMPT:")
print(bike_ex['text'])



print("PREPROCESSING — REMOVING BOS TOKEN")


print("\n   Why we remove BOS token:")
print(f"   BOS token ID = {tokenizer.bos_token_id}")
print(f"   Tokenizer vocab = {tokenizer.vocab_size:,}")
print(f"   BOS ({tokenizer.bos_token_id}) > vocab ({tokenizer.vocab_size}) = CRASH!")
print(f"   Solution: add_special_tokens=False removes BOS ")

VOCAB_SIZE = model.config.vocab_size

def preprocess(example):
    tokens = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        add_special_tokens=False,  # removes BOS token 128000
    )
    tokens['input_ids'] = [
        min(t, VOCAB_SIZE - 1) for t in tokens['input_ids']
    ]
    return tokens

print("\n   Processing all examples...")
train_processed = ds['train'].map(
    preprocess,
    remove_columns=ds['train'].column_names,
    batched=False
)
val_processed = ds['validation'].map(
    preprocess,
    remove_columns=ds['validation'].column_names,
    batched=False
)

# Verify
max_id    = max(max(ex['input_ids']) for ex in train_processed)
has_bos   = any(tokenizer.bos_token_id in ex['input_ids'] for ex in train_processed)
bad_count = sum(1 for ex in train_processed
                for t in ex['input_ids'] if t >= VOCAB_SIZE)

print(f"\n PREPROCESSING RESULTS:")
print(f"   Train processed : {len(train_processed):,}")
print(f"   Val processed   : {len(val_processed):,}")
print(f"   Max token ID    : {max_id:,} (vocab={VOCAB_SIZE:,}) ")
print(f"   Has BOS token   : {has_bos} ")
print(f"   Bad token IDs   : {bad_count} ")

print(f"\n BEFORE preprocessing (first 10 token IDs):")
raw = tokenizer(ds['train'][0]['text'], add_special_tokens=True)
print(f"   {raw['input_ids'][:10]}")
print(f"    starts with {raw['input_ids'][0]} = BOS = DANGEROUS!")

print(f"\n AFTER preprocessing (first 10 token IDs):")
print(f"   {train_processed[0]['input_ids'][:10]}")
print(f"    starts with {train_processed[0]['input_ids'][0]} = safe regular token ")


print(f"   Dataset ready for training!")
print(f"   {len(train_processed):,} clean training examples")
print(f"   {len(val_processed):,} clean validation examples")


 Loading dataset from HuggingFace...

 Dataset loaded!
DATASET OVERVIEW

 SIZE:
   Train examples : 6,570
   Val examples   : 730
   Total          : 7,300

 COLUMNS:
   → text
   → db_id
   → question
   → sql
   → source
ONE COMPLETE TRAINING EXAMPLE

 SOURCE: spider
   (spider = general SQL | bikeshare = my domain)

 DB_ID: hospital_1
   (which database this example belongs to)

 QUESTION:
   Find the names of nurses who are nursing an undergoing treatment.

 SQL (correct answer):
   SELECT DISTINCT T2.name FROM undergoes AS T1 JOIN nurse AS T2 ON T1.AssistingNurse  =  T2.EmployeeID

 FULL TEXT (what model sees):
### Instruction:
You are an expert SQL assistant. Given a database schema and a natural language question, write a correct SQL query. Only output the SQL query, no explanation.

### Schema:
CREATE TABLE Physician (
  EmployeeID number,
  Name text,
  Position text,
  SSN number
);

CREATE TABLE Department (
  DepartmentID number,
  Name text,
  Head number
);

CREATE TABLE

Map:   0%|          | 0/730 [00:00<?, ? examples/s]


 PREPROCESSING RESULTS:
   Train processed : 6,570
   Val processed   : 730
   Max token ID    : 127,723 (vocab=128,256) 
   Has BOS token   : False 
   Bad token IDs   : 0 

 BEFORE preprocessing (first 10 token IDs):
   [128000, 14711, 30151, 512, 2675, 527, 459, 6335, 8029, 18328]
    starts with 128000 = BOS = DANGEROUS!

 AFTER preprocessing (first 10 token IDs):
   [14711, 30151, 512, 2675, 527, 459, 6335, 8029, 18328, 13]
    starts with 14711 = safe regular token 
   Dataset ready for training!
   6,570 clean training examples
   730 clean validation examples


In [7]:
# LoRA Configuration  
from peft import LoraConfig, get_peft_model


print("\nMODEL BEFORE LoRA:")
total_before = sum(p.numel() for p in model.parameters())
train_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total params     : {total_before:,}")
print(f"   Trainable params : {train_before:,}")
print(f"   Frozen params    : {total_before - train_before:,}")


print("\nLoRA CONFIGURATION:")
print(f"   r (rank)         : {LORA_R}")
print(f"   lora_alpha       : {LORA_ALPHA}")
print(f"   lora_dropout     : {LORA_DROPOUT}")
print(f"   bias             : none")
print(f"   task_type        : CAUSAL_LM")
print(f"\n   Target modules (which layers get LoRA adapters):")
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules
)


print("\nApplying LoRA adapters to model...")
model = get_peft_model(model, lora_config)
print("LoRA applied!")


print("\nMODEL AFTER LoRA:")
total_after = sum(p.numel() for p in model.parameters())
train_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_after = total_after - train_after

print(f"   Total params     : {total_after:,}")
print(f"   Trainable params : {train_after:,}")
print(f"   Frozen params    : {frozen_after:,}")
print(f"   Trainable %      : {train_after/total_after*100:.2f}%")


print(f"\nMEMORY AFTER LoRA:")
print(f"   Model memory     : {model.get_memory_footprint()/1e9:.2f} GB")
print(f"   Used VRAM        : {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"   Free VRAM        : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.1f} GB")


print(f"\n LoRA ADAPTER DETAILS:")
print(f"   Each target layer gets TWO small matrices:")
print(f"   Matrix A : layer_dim × {LORA_R}  (down projection)")
print(f"   Matrix B : {LORA_R} × layer_dim  (up projection)")
print(f"\n   Example for q_proj (3072 × 3072):")
print(f"   Original : 3072 × 3072 = 9,437,184 params (FROZEN)")
print(f"   LoRA A   : 3072 × {LORA_R}  =   {3072*LORA_R:,} params (trainable)")
print(f"   LoRA B   : {LORA_R} × 3072  =   {LORA_R*3072:,} params (trainable)")
print(f"   Total LoRA for q_proj: {2*LORA_R*3072:,} params")
print(f"   vs Original          : {3072*3072:,} params")
print(f"   Reduction            : {(1 - 2*LORA_R*3072/(3072*3072))*100:.1f}% fewer params!")



print(f"\n   Q (Quantization):")
print(f"   Base model compressed 32-bit to 4-bit")
print(f"   Memory: 13GB to 2.2GB (saved 83%)")
print(f"")
print(f"   LoRA (Low Rank Adaptation):")
print(f"   Added {train_after:,} trainable parameters")
print(f"   Only {train_after/total_after*100:.2f}% of total params train")
print(f"   Base model stays completely frozen")
print(f"")
print(f"   Combined (QLoRA):")
print(f"   Train a 3B model on a single GPU")
print(f"   In ~45 minutes")
print(f"   With $0 cost")
print(f"   Getting results close to full fine-tuning!")



MODEL BEFORE LoRA:
   Total params     : 1,803,463,680
   Trainable params : 394,177,536
   Frozen params    : 1,409,286,144

LoRA CONFIGURATION:
   r (rank)         : 64
   lora_alpha       : 128
   lora_dropout     : 0.05
   bias             : none
   task_type        : CAUSAL_LM

   Target modules (which layers get LoRA adapters):

Applying LoRA adapters to model...
LoRA applied!

MODEL AFTER LoRA:
   Total params     : 1,900,719,104
   Trainable params : 97,255,424
   Frozen params    : 1,803,463,680
   Trainable %      : 5.12%

MEMORY AFTER LoRA:
   Model memory     : 2.59 GB
   Used VRAM        : 2.88 GB
   Free VRAM        : 44.8 GB

 LoRA ADAPTER DETAILS:
   Each target layer gets TWO small matrices:
   Matrix A : layer_dim × 64  (down projection)
   Matrix B : 64 × layer_dim  (up projection)

   Example for q_proj (3072 × 3072):
   Original : 3072 × 3072 = 9,437,184 params (FROZEN)
   LoRA A   : 3072 × 64  =   196,608 params (trainable)
   LoRA B   : 64 × 3072  =   196,608 pa

In [4]:
# Run this in a cell first, then RESTART THE KERNEL
!pip install --upgrade --force-reinstall --no-cache-dir \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    unsloth_zoo \
    transformers \
    trl \
    peft

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-y_qwxyec/unsloth_d082a7fae9db4391b7381e50cb3d1723
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-y_qwxyec/unsloth_d082a7fae9db4391b7381e50cb3d1723
  Resolved https://github.com/unslothai/unsloth.git to commit 07aabf45c03725046abcb3b28b874224dc6708f8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 94.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 367.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 333.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 415.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 367.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 322.0 MB/s eta 

In [1]:
# unsloth MUST be imported first, before anything else
import unsloth
from unsloth import FastLanguageModel

import torch
from huggingface_hub import login
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

HF_TOKEN = "Your Key Here" 
BASE_MODEL   = "meta-llama/Llama-3.2-3B-Instruct"
DATASET_NAME = "text2sqldata"
HUB_OUT      = "text2sql-llama3-qlora"
RUN_NAME     = "text2sql-run-1"
EPOCHS       = 3
LR           = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
MAX_SEQ_LEN  = 512
SAVE_STEPS   = 100
LOG_STEPS    = 10
LORA_R       = 64
LORA_ALPHA   = 128

login(token=HF_TOKEN)

# Model + Tokenizer 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = torch.float16,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.0,
    bias           = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    use_gradient_checkpointing = False,
    random_state               = 42,
)

#  Dataset 
ds       = load_dataset(DATASET_NAME)
train_ds = ds['train'].select_columns(['text'])
val_ds   = ds['validation'].select_columns(['text'])

#  Trainer 
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args = SFTConfig(
        output_dir                  = f"./{RUN_NAME}",
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = GRAD_ACCUM,
        max_seq_length              = MAX_SEQ_LEN,
        dataset_text_field          = "text",
        packing                     = False,
        learning_rate               = LR,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.03,
        weight_decay                = 0.001,
        optim                       = "paged_adamw_32bit",
        max_grad_norm               = 0.3,
        bf16                        = False,
        fp16                        = True,
        logging_steps               = LOG_STEPS,
        report_to                   = "none",
        save_strategy               = "steps",
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 3,
        eval_strategy               = "steps",
        eval_steps                  = SAVE_STEPS,
        push_to_hub                 = True,
        hub_model_id                = HUB_OUT,
        hub_strategy                = "every_save",
        hub_private_repo            = False,
        hub_token                   = HF_TOKEN,
        remove_unused_columns       = True,
        gradient_checkpointing      = False,
    )
)

print("Training Started")
trainer.train()
print("Training Completed")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.4.0' currently installed).
  from pandas.core import (


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA L40. Num GPUs = 2. Max memory: 44.394 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Training Started


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,570 | Num Epochs = 3 | Total steps = 1,233
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


Step,Training Loss,Validation Loss
100,0.196877,0.201256
200,0.133791,0.155974
300,0.139745,0.144265
400,0.122508,0.135704
500,0.092935,0.129639
600,0.102355,0.123794
700,0.079545,0.120456
800,0.088597,0.115994
900,0.066926,0.120835
1000,0.067510,0.118379


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training Completed


In [2]:
print("Pushing final model to HuggingFace Hub...")

trainer.model.push_to_hub(
    HUB_OUT,
    token=HF_TOKEN
)
tokenizer.push_to_hub(
    HUB_OUT,
    token=HF_TOKEN
)

print(f"Model pushed!")
print(f"View at: https://huggingface.co/Krupa1420/text2sql-llama3-qlora")

Pushing final model to HuggingFace Hub...


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.YOUR_HF_TOKEN_HERE|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/Krupa1420/text2sql-llama3-qlora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model pushed!
View at: https://huggingface.co/Krupa1420/text2sql-llama3-qlora


In [3]:
import os

# Check what checkpoints were saved
checkpoint_dir = f"./{RUN_NAME}"
if os.path.exists(checkpoint_dir):
    checkpoints = sorted([
        d for d in os.listdir(checkpoint_dir)
        if d.startswith('checkpoint')
    ])
    print(f"Available checkpoints: {checkpoints}")
else:
    print("No local checkpoints found")

# Also check HuggingFace Hub
print(f"\nModel on HuggingFace: https://huggingface.co/{HUB_OUT}")

Available checkpoints: ['checkpoint-1100', 'checkpoint-1200', 'checkpoint-1233']

Model on HuggingFace: https://huggingface.co/Krupa1420/text2sql-llama3-qlora
